In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [16]:
def add_user_message(messages, text):
    user_message={"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params={
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

# 시스템 메시지가 설정된 경우에만 params에 system을 포함하게끔 하는 if 함수
    if system:
        params["system"]= system
# (**)는 params 주머니 안의 내용물을 자동으로 꺼내서 API 양식에 맞게 채워주는 규칙임     
    message=client.messages.create(**params)
    return message.content[0].text

In [17]:
#시스템 프롬프트를 입력하지 않았을 때의 클로드 답변
messages=[]

add_user_message(messages, "회사가 차입금 약정(Covenant)상의 부채비율 조건을 위반했어. 은행이 당장 갚으라고 독촉하지는 않았지만, 재무상태표일(12/31)까지 은행으로부터 상환유예(Waiver)를 서면으로 받지는 못했어. 이거 비유동부채로 유지해도 돼?")

answer = chat(messages)

add_assistant_message(messages, answer)

print(answer)

# 차입금 Covenant 위반 시 부채 분류

## 결론: **유동부채로 재분류해야 합니다** ❌

---

## 근거 (K-IFRS 제1001호 / IAS 1)

### 핵심 원칙
> 보고기간말 현재 **장기 차입약정을 위반**한 경우, 대여자가 **즉시 상환을 요구할 수 있는 권리**가 생기므로 → **유동부채로 분류**

---

## 판단 기준 체계

```
Covenant 위반 발생
        ↓
보고기간말(12/31) 기준으로 판단
        ↓
┌─────────────────────────────────────┐
│ 은행이 12/31까지 Waiver를 서면 부여? │
└─────────────────────────────────────┘
        ↓ NO
   ┌────────────────────────────────┐
   │ 유동부채 분류 (강제)           │
   │ - 은행의 독촉 여부 무관        │
   │ - 실제 상환 요구 여부 무관     │
   └────────────────────────────────┘
        ↓ YES (12/31 이전 Waiver)
   비유동부채 유지 가능
```

---

## 자주 혼동하는 포인트

| 상황 | 분류 |
|------|------|
| 위반 + 12/31 이전 Waiver 획득 | ✅ 비유동 유지 |
| 위반 + 12/31 이후 Waiver 획득 | ❌ 유동 분류 |
| 위반 + 은행이 독촉 안 함 (Waiver 없음) | ❌ 유동 분류 |
| 위반 + 협의 중 (Waiver 미확정) | ❌ 유동 분류 |

---

## 왜 엄격하게 보는가?

```
법적 권리 관점
━━━━━━━━━━━━━
Covenant 위반 시점부터
은행은 "즉시 상환 청구권" 보유

→ 회사가 12개월 내 결제를 
  회피할 능력이 없는 상태
  
→ IFRS는 법적 실질(Legal Substance) 우선
  (은행의 의도/관행 불문)
```

---

## 실무 처리 

In [18]:
#시스템 프롬프트를 입력했을 때의 클로드 답변
messages=[]

system = """
너는 대형 회계법인(Big 4)에서 10년 차 이상의 경력을 가진 '감사 전문가'야. 
회계사(사용자)가 차입금 관련 상황이나 데이터를 주면, 일반적인 설명을 넘어 '감사인의 판단'을 내리고 이는 해당 법인의 감사조서에 반영될 예정이야.

답변 시 다음의 [지침]을 반드시 준수해:

1. [기준서 근거]: 판단의 근거로 KGAAP(일반기업회계기준)의 구체적인 기준서 번호와 관련 문구를 반드시 인용할 것.
2. [감사 의구심]: 감사인이 추가로 확인해야 할 '잠재적 리스크'나 '필수 추가 확인 서류'의 목록을 2~3가지 정도로 간추려서 제시할 것. 향후 조서에 기입하고 확인여부를 체크할 예정
3. [전문적 어조]: 친절한 비서 말투가 아닌, 냉철하고 간결한 감사인의 전문 용어를 사용할 것.
4. [조서 양식 출력]: 답변 마지막에는 반드시 아래 양식으로 요약할 것:
   - 수행한 감사 절차 (Audit Procedures):
   - 감사 결론 (Conclusion):
"""

add_user_message(messages, "회사가 차입금 약정(Covenant)상의 부채비율 조건을 위반했어. 은행이 당장 갚으라고 독촉하지는 않았지만, 재무상태표일(12/31)까지 은행으로부터 상환유예(Waiver)를 서면으로 받지는 못했어. 이거 비유동부채로 유지해도 돼?")


answer = chat(messages)

add_assistant_message(messages, answer)

print(answer)

# 차입금 Covenant 위반 시 부채 분류 검토

## 결론: **유동부채로 재분류 필요**

---

## 근거 (K-IFRS 제1001호 / IAS 1)

### 핵심 원칙
> 보고기간 말 현재 **기존 대출약정을 위반**한 경우, 채권자가 **즉시 상환을 요구할 수 있는 권리**가 생기므로 → **유동부채 분류**

```
K-IFRS 1001호 문단 74~75
```

---

## 세부 판단 로직

| 조건 | 내용 | 귀사 상황 |
|------|------|-----------|
| Covenant 위반 발생 | 보고기간 말 이전 위반 | ✅ 위반 |
| 채권자 즉시상환 요구권 | 위반 시 기한이익상실 조항 | ✅ 해당 |
| **Waiver 수령** | **보고기간 말 이전** 서면 취득 | ❌ **미취득** |
| 보고기간 후 Waiver 수령 | 12/31 이후 취득 | → **효과 없음** |

---

## 왜 "독촉 안 했다"는 것이 중요하지 않은가?

- **실제로 상환 요구를 했는지 여부가 아니라**
- 채권자에게 **즉시 상환을 요구할 수 있는 법적 권리**가 존재하는지가 기준
- 은행이 "선의로 기다려주는 것"은 회계처리와 무관

---

## Waiver 시점에 따른 분류

```
📅 Timeline

  위반 발생        12/31          재무제표
     │          (보고기간말)      발행일
     ▼               ▼              ▼
─────●───────────────●──────────────●────▶
     
     [Waiver 수령 시점별 처리]
     
     ① 12/31 이전 수령  →  ✅ 비유동 유지 가능
     ② 12/31 이후 수령  →  ❌ 유동 분류 + 주석공시
                              (수정을 요하지 않는 후속사건)
```

---

## 실무 처리 사항

### 1️⃣ 재무상태표
- 해당 차입금 → 